# 2단계 — 롤링 여부 / GP·Theta 설명 / Lasso 튜닝

- **3. 롤링 여부**: 회귀/GP/ARIMA/Theta·Holt가 다중스텝 예측에서 롤링을 쓰는지 코드로 증명
- **4. GP 설명**: 입력=시간 인덱스, 롤링이 불필요한 이유
- **5. Theta 설명**: 실제 코드 + 알고리즘적 근거
- **6. Lasso 튜닝**: 튜닝 여부 확인 + 정량 비교

대표변수: `X3_l1`

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

H = 4
TARGET_VAR = "X3_l1"
LABELS = {
    "X1_l1": "X1", "X2": "X2", "X3_l1": "X3",
    "X4": "X4", "X5_l1": "X5", "X6_l1": "X6",
}

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", TARGET_VAR]].dropna()[TARGET_VAR].values.astype(float)
n = len(vals)
print(f"대표변수: {LABELS[TARGET_VAR]} ({TARGET_VAR}), 데이터 {n}개 분기")


대표변수: X3 (X3_l1), 데이터 60개 분기


## 3. 롤링 사용여부 확인

| 모델 계열 | 롤링(재귀) 사용? | 이유 |
|---|---|---|
| 회귀(OLS/Ridge/Lasso/ElasticNet)/RF/kNN | **씀** | lag(과거값)를 입력으로 쓰는데, h=2 이상은 미래값이라 직전 예측값을 lag 입력에 대신 넣어야 함 |
| ARIMA | **씀(내부적으로)** | AR항이 있으면 h=2 예측에 h=1 예측값을 대신 넣어야 함 — 수학적으로 필연. 다만 라이브러리 호출 1번 안에서 자동 처리됨 |
| GP | 안 씀 | 입력이 "몇 번째 시점인가"(시간 인덱스)뿐이라 미래 시점 인덱스는 이미 다 알고 있음 — 예측값을 입력으로 재사용할 필요가 없음 |
| Theta/Holt | 안 씀 | "마지막 시점 수준 + h×추세"라는 닫힌 형태 공식으로 h를 그냥 대입해서 한 번에 계산 |

아래에서 회귀/ARIMA/Theta 순서로 코드 증명. GP는 다루는 내용이 많아 4번 섹션에서 별도로 자세히 본다.


### 3-1. 회귀/RF/kNN -- 롤링 증명

`hist.append(yp)`로 방금 만든 예측값을 다음 스텝 입력으로 재사용한다.

In [2]:
from sklearn.linear_model import LinearRegression

LAG = 2

def _lag_xy(values, lag=LAG):
    X = np.array([values[i:i + lag] for i in range(len(values) - lag)])
    y = values[lag:]
    return X, y


def _recursive_forecast_verbose(model_fit_fn, values, h=H, lag=LAG):
    X, y = _lag_xy(values, lag)
    model = model_fit_fn(X, y)
    hist = list(values[-lag:])
    print(f"  시작 hist(마지막 {lag}개 실측값): {[round(v, 2) for v in hist]}")
    out = []
    for step in range(h):
        x = np.array(hist[-lag:]).reshape(1, -1)
        yp = model.predict(x)[0]
        out.append(yp)
        hist.append(yp)  # <- 핵심: 방금 만든 예측값을 다음 입력으로 재사용(롤링)
        print(f"  h={step + 1}: 입력={[round(v, 2) for v in x[0]]} -> 예측={round(yp, 2)}  (hist에 추가됨)")
    return np.array(out)


print("[회귀(OLS) 롤링 증명]")
fc = _recursive_forecast_verbose(lambda X, y: LinearRegression().fit(X, y), vals, H)
print("최종 예측:", np.round(fc, 2))


[회귀(OLS) 롤링 증명]
  시작 hist(마지막 2개 실측값): [np.float64(983.84), np.float64(1009.09)]
  h=1: 입력=[np.float64(983.84), np.float64(1009.09)] -> 예측=988.66  (hist에 추가됨)
  h=2: 입력=[np.float64(1009.09), np.float64(988.66)] -> 예측=950.14  (hist에 추가됨)
  h=3: 입력=[np.float64(988.66), np.float64(950.14)] -> 예측=907.49  (hist에 추가됨)
  h=4: 입력=[np.float64(950.14), np.float64(907.49)] -> 예측=867.24  (hist에 추가됨)
최종 예측: [988.66 950.14 907.49 867.24]


### 3-2. ARIMA -- 롤링 증명

AR항이 있으면 h=2 예측에 h=1 예측값을 대신 써야 한다(수학적 필연). 통제된 AR(1)로 수동 재귀계산 vs pmdarima `predict()`를 비교해 검증한다.

In [3]:
import pmdarima as pm

rng = np.random.default_rng(0)
n_demo = 60
phi_true = 0.6
y_demo = [0.0]
for _ in range(n_demo - 1):
    y_demo.append(phi_true * y_demo[-1] + rng.normal(0, 1))
y_demo = np.array(y_demo)
y_c = y_demo - y_demo.mean()  # 평균을 0으로 맞춰 절편(intercept) 해석 문제를 없앰

m = pm.auto_arima(y_c, seasonal=False, d=0, max_p=1, max_q=0, start_p=1, start_q=0,
                   suppress_warnings=True, with_intercept=False)
phi_hat = m.arparams()[0]
print(f"추정된 AR(1) 계수(phi_hat) = {phi_hat:.4f} (실제값 {phi_true})")

# 수동 재귀 계산: yhat(n+h) = phi * yhat(n+h-1), 매 스텝 직전 예측값을 그대로 다음 입력으로 씀
prev = y_c[-1]
manual_fc = []
for h in range(1, H + 1):
    prev = phi_hat * prev
    manual_fc.append(prev)
    print(f"  h={h}: yhat = phi_hat * (h={h - 1}의 예측값) = {prev:.4f}")
manual_fc = np.array(manual_fc)

lib_fc = m.predict(n_periods=H)
print("\n수동 재귀 계산:      ", np.round(manual_fc, 6))
print("pmdarima predict():  ", np.round(lib_fc, 6))
print("완전히 일치하는가:", np.allclose(manual_fc, lib_fc, atol=1e-6))


추정된 AR(1) 계수(phi_hat) = 0.7600 (실제값 0.6)
  h=1: yhat = phi_hat * (h=0의 예측값) = -0.6626
  h=2: yhat = phi_hat * (h=1의 예측값) = -0.5036
  h=3: yhat = phi_hat * (h=2의 예측값) = -0.3827
  h=4: yhat = phi_hat * (h=3의 예측값) = -0.2909

수동 재귀 계산:       [-0.662586 -0.503567 -0.382712 -0.290862]
pmdarima predict():   [-0.662586 -0.503567 -0.382712 -0.290862]
완전히 일치하는가: True


**결론**: 수동 계산과 pmdarima 결과가 소수점까지 일치 -- ARIMA 내부에서 롤링(재귀)이 실제로 일어난다는 직접 증거.

### 3-3. Theta/Holt -- 롤링 안 씀 증명

추세가 없는 SES 성분은 h와 무관하게 항상 같은 값이어야 한다 -- 검증한다.

In [4]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

slope = np.polyfit(np.arange(n), vals, 1)[0]
ses_fc = ExponentialSmoothing(vals, trend=None, initialization_method="estimated").fit().forecast(H)
drift = (slope / 2) * np.arange(1, H + 1)
theta_fc = ses_fc + drift

print("SES 성분(h=1~4):     ", np.round(ses_fc, 3))
print("  -> h에 상관없이 전부 동일한가:", np.allclose(ses_fc, ses_fc[0]))
print("드리프트 성분(h=1~4):", np.round(drift, 3))
print("최종 Theta 예측:      ", np.round(theta_fc, 3))


SES 성분(h=1~4):      [1009.088 1009.088 1009.088 1009.088]
  -> h에 상관없이 전부 동일한가: True
드리프트 성분(h=1~4): [ 4.968  9.935 14.903 19.87 ]
최종 Theta 예측:       [1014.055 1019.023 1023.991 1028.958]


**결론**: SES 성분은 h=1~4 전부 동일값. 차이는 드리프트항(`slope/2 * h`)뿐 -- 이전 예측값 재사용 없음. Holt도 h를 공식에 바로 대입하는 구조.

## 4. GP 모델 추가 설명

### 4-1. 입력변수 = "시간 인덱스"

GP는 lag(과거값) 대신 "몇 번째 시점인가"만 입력으로 쓴다.

In [5]:
Xt = np.arange(n).reshape(-1, 1) / n
print("GP에 실제로 들어가는 입력(Xt) 앞 5개:", Xt[:5].ravel())
print("GP에 실제로 들어가는 입력(Xt) 뒤 5개:", Xt[-5:].ravel())
print("입력의 shape:", Xt.shape, "-- 컬럼이 1개뿐이고, 이 컬럼은 오직 '몇 번째 시점인가'다(y값 자체는 입력에 없음)")


GP에 실제로 들어가는 입력(Xt) 앞 5개: [0.         0.01666667 0.03333333 0.05       0.06666667]
GP에 실제로 들어가는 입력(Xt) 뒤 5개: [0.91666667 0.93333333 0.95       0.96666667 0.98333333]
입력의 shape: (60, 1) -- 컬럼이 1개뿐이고, 이 컬럼은 오직 '몇 번째 시점인가'다(y값 자체는 입력에 없음)


### 4-2. 롤링이 필요 없는 이유

미래 시점 인덱스는 이미 알고 있으므로, h=1~4를 한 번에/개별로 예측해도 결과가 같아야 한다 -- 검증한다.

In [6]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

kernel = ConstantKernel(1.0) * Matern(1.0, nu=1.5) + WhiteKernel(1.0)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=2, alpha=1e-6, random_state=0).fit(Xt, vals)

# (A) h=1~4를 한 번에 조인트로 예측
Xf_joint = np.arange(n, n + H).reshape(-1, 1) / n
mean_joint, _ = gp.predict(Xf_joint, return_std=True)

# (B) h=1~4를 하나씩 따로(다른 스텝의 예측값을 전혀 모르는 채로) 예측
mean_separate = []
for h in range(1, H + 1):
    xf = np.array([[(n + h - 1) / n]])
    mh, _ = gp.predict(xf, return_std=True)
    mean_separate.append(mh[0])
mean_separate = np.array(mean_separate)

print("한 번에 h=1~4 조인트 예측:", np.round(mean_joint, 4))
print("하나씩 따로따로 예측:      ", np.round(mean_separate, 4))
print("완전히 같은가(=롤링이 필요 없다는 증거):", np.allclose(mean_joint, mean_separate))


한 번에 h=1~4 조인트 예측: [984.2396 941.6973 890.7308 840.339 ]
하나씩 따로따로 예측:       [984.2396 941.6973 890.7308 840.339 ]
완전히 같은가(=롤링이 필요 없다는 증거): True


**결론**: 두 방식 결과가 완전히 같다 -- GP는 h=1 예측값을 h=2 계산에 쓰지 않는다는 증거.

## 5. Theta 모델 추가 설명

```python
def fc_theta(values, h=H):
    slope = np.polyfit(np.arange(len(values)), values, 1)[0]  # 전체 계열의 선형추세 기울기(OLS)
    ses_fc = ExponentialSmoothing(values, trend=None, initialization_method="estimated").fit().forecast(h)  # SES: 추세 없이 평평한 예측(h와 무관하게 동일)
    return ses_fc + (slope / 2) * np.arange(1, h + 1)  # SES + 기울기 절반*h -- Theta의 증명된 동치식
```

정식 Theta(Assimakopoulos & Nikolopoulos, 2000)는 theta=0(추세선)·theta=2(SES) 두 선을 50:50 평균한다. Hyndman & Billah(2003)이 이게 수학적으로 **"SES + 선형기울기 절반"**과 동치임을 증명했다 -- 위 코드는 그 증명된 동치식 그대로다(축약판 아님).

## 6. 회귀 Lasso 모델 튜닝 점검

### 기존 튜닝 여부

```python
LassoCV(alphas=np.logspace(-3, 3, 13), max_iter=5000)
```

이미 튜닝돼 있었다 -- **그리드서치**(랜덤/베이지안 아님)로 13개 alpha 후보(로그간격 0.001~1000)를 **5-fold 교차검증**해서 평균 MSE가 가장 낮은 걸 채택. origin마다 그 시점 데이터로만 매번 새로 탐색(미래 누출 없음).

### 안 튜닝했으면?

디폴트 `Lasso(alpha=1.0)`과 비교한다.

In [7]:
from sklearn.linear_model import Lasso, LassoCV

MIN_TRAIN = 20


def fc_lasso_untuned(values, h=H):
    return _recursive_forecast_quiet(lambda X, y: Lasso(alpha=1.0, max_iter=5000).fit(X, y), values, h)


def fc_lasso_tuned(values, h=H):
    return _recursive_forecast_quiet(lambda X, y: LassoCV(alphas=np.logspace(-3, 3, 13), max_iter=5000).fit(X, y), values, h)


def _recursive_forecast_quiet(model_fit_fn, values, h=H, lag=LAG):
    X, y = _lag_xy(values, lag)
    model = model_fit_fn(X, y)
    hist = list(values[-lag:])
    out = []
    for _ in range(h):
        x = np.array(hist[-lag:]).reshape(1, -1)
        yp = model.predict(x)[0]
        out.append(yp)
        hist.append(yp)
    return np.array(out)


var_std = vals.std()
rows = []
for origin in range(MIN_TRAIN, n - H + 1):
    train, actual = vals[:origin], vals[origin:origin + H]
    fc_u = fc_lasso_untuned(train, H)
    fc_t = fc_lasso_tuned(train, H)
    rows.append({
        "origin": origin,
        "미튜닝 정규화오차": np.mean(np.abs(actual - fc_u)) / var_std,
        "튜닝(LassoCV) 정규화오차": np.mean(np.abs(actual - fc_t)) / var_std,
    })
lasso_df = pd.DataFrame(rows)
mean_u = lasso_df["미튜닝 정규화오차"].mean()
mean_t = lasso_df["튜닝(LassoCV) 정규화오차"].mean()
improve_pct = (mean_u - mean_t) / mean_u * 100

print(f"미튜닝 Lasso(alpha=1.0 고정)   평균 정규화오차: {mean_u:.4f}")
print(f"튜닝(LassoCV, 13개 alpha탐색) 평균 정규화오차: {mean_t:.4f}")
print(f"튜닝으로 개선된 비율: {improve_pct:.1f}%")


미튜닝 Lasso(alpha=1.0 고정)   평균 정규화오차: 0.6214
튜닝(LassoCV, 13개 alpha탐색) 평균 정규화오차: 0.6295
튜닝으로 개선된 비율: -1.3%


**결론(예상외)**: 튜닝(0.6295)이 미튜닝(0.6214)보다 오히려 1.3% 나쁘다.

의심 원인: origin당 표본이 18개뿐이라 5-fold CV가 폴드당 3~4개로 너무 적어 alpha 선택이 노이즈에 취약.

**주의**: 차이가 작고(origin 37개) 통계적 유의성은 미검증 -- "튜닝하면 무조건 낫다"는 전제가 이 표본 크기엔 안 맞을 수 있다는 정도로만 해석한다.